# 1ª entrega — Análise Exploratória de Dados (AED) comentada
**Aluno(a):** _seu nome_ · **Disciplina:** Python Machine Learning

**Pergunta do projeto:** a partir de biomarcadores, é possível (1) classificar o paciente em `BCG`, `cancer`, `imunodeficiencia` ou `leucemia`, (2) prever o valor de `IFN-γ` e (3) encontrar grupos naturais (clusters)?

**Critério de sucesso (defina o seu):** ex.: F1-macro ≥ 0,75 na classificação; R² ≥ 0,70 na regressão.

> Em cada bloco **✍️ Comentário**, escreva com suas palavras o que o resultado mostra. É isso que transforma a AED em *relatório comentado*.

In [ ]:
# No Colab, rode esta linha uma vez:
# !pip install ydata-profiling
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../dados/biomarcadores.csv")
df.shape

## 1. Visão geral

In [ ]:
df.head()

In [ ]:
df.info()

✍️ **Comentário:** quantas observações e variáveis? Quantas são numéricas, lógicas e categóricas?

## 2. Valores faltantes e constantes

In [ ]:
faltantes = df.isna().sum()
faltantes[faltantes > 0].sort_values(ascending=False)

In [ ]:
constantes = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
constantes

✍️ **Comentário:** quais variáveis têm faltantes e em que proporção? Quais são constantes e por que devem ser removidas?

## 3. Variável alvo — balanceamento das classes

In [ ]:
df["classe"].value_counts(normalize=True).round(3)

In [ ]:
sns.countplot(data=df, x="classe")
plt.title("Distribuição das classes")
plt.show()

✍️ **Comentário:** as classes estão balanceadas? Seria necessário SMOTE?

## 4. Medidas resumo e distribuições

In [ ]:
df.describe().T

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df["PPD"], ax=ax[0]).set_title("PPD (original)")
sns.histplot(np.log1p(df["PPD"]), ax=ax[1]).set_title("log(1 + PPD)")
plt.show()

✍️ **Comentário:** a PPD é assimétrica? O que a transformação log mudou?

## 5. Variáveis × classe

In [ ]:
for var in ["TP53", "IFN-γ", "CD3_2", "CD8", "CD7"]:
    sns.boxplot(data=df, x="classe", y=var)
    plt.title(var)
    plt.show()

In [ ]:
pd.crosstab(df["FOXP3"], df["classe"], normalize="columns").round(2)

✍️ **Comentário:** quais biomarcadores separam bem as classes? Qual classe se destaca em IFN-γ?

## 6. Correlação

In [ ]:
corr = df.select_dtypes("number").corr(method="spearman")
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.show()

In [ ]:
# Pares muito correlacionados (|r| > 0,7) -> manter só um de cada par
pares = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
pares[pares.abs() > 0.7].sort_values()

✍️ **Comentário:** CD34, CD19_1, CD20_1, CD22, CD3_1 e CD7 formam um bloco correlacionado. Por que faz sentido manter só uma delas (aqui, CD7)?

## 7. Relatório automático (ydata-profiling)

In [ ]:
from ydata_profiling import ProfileReport

perfil = ProfileReport(df, title="AED - Biomarcadores", explorative=True)
perfil.to_file("relatorio_aed.html")
# perfil.to_notebook_iframe()  # para ver dentro do notebook

## 8. Conclusões da AED
✍️ Resuma em 4–6 tópicos:
- Qualidade dos dados (faltantes, constantes, duplicados) e como tratou;
- Balanceamento das classes;
- Transformações feitas (ex.: `PPD_log`);
- Variáveis selecionadas para os modelos: `TP53, IFN-γ, CD3_2, CD8, PPD_log, CD7, FOXP3`;
- Hipóteses para a modelagem.